# Tutorial 14 — Video, unified multimodal systems, and generative frontiers

[Lecture notes](../lectures/14_video_omni.md) · [Exercise sheet](../exercises/14_video_omni.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Generate a moving-square video as pixel arrays, expose frame-sampling failures, and inspect forward Gaussian noising.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Generate a short video as arrays

A white square moves left to right. Frame 7 contains a brief red marker. Sampling may preserve motion direction but miss the marker.

Before running: predict the output or the invariant being checked.


In [2]:
video=np.zeros((24,16,32,3),dtype=np.float32)
for f in range(24): video[f,6:10,f:f+4,:]=1
video[7,0:4,0:4]=[1,0,0]
def event_seen(frames):
    return bool(np.any((frames[...,0]>.9)&(frames[...,1]<.1)&(frames[...,2]<.1)))
def direction(frames):
    positions=[]
    for im in frames:
        mask=(im.min(axis=-1)>.9)
        positions.append(np.where(mask)[1].mean())
    return "right" if positions[-1]>positions[0] else "left" if positions[-1]<positions[0] else "stationary"
print("video",video.shape,"full event",event_seen(video),"direction",direction(video))
assert event_seen(video) and direction(video)=="right"


video (24, 16, 32, 3) full event True direction right


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Sampling phase and temporal reversal

Offsets matter. A result from one favorable sampling offset cannot establish robust event detection.

Before running: predict the output or the invariant being checked.


In [3]:
for stride in [1,2,4,8]:
    rates=[]
    for offset in range(stride):
        sampled=video[offset::stride]; rates.append(event_seen(sampled))
    print("stride",stride,"fraction of offsets detecting event",np.mean(rates),"frames at offset 0",len(video[::stride]))
print("reversed direction",direction(video[::-1]))
assert direction(video[::-1])=="left"
assert not event_seen(video[::4])


stride 1 fraction of offsets detecting event 1.0 frames at offset 0 24
stride 2 fraction of offsets detecting event 0.5 frames at offset 0 12
stride 4 fraction of offsets detecting event 0.25 frames at offset 0 6
stride 8 fraction of offsets detecting event 0.125 frames at offset 0 3
reversed direction left


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Forward noising and context estimates

This is the forward Gaussian corruption equation x_t=sqrt(alpha_bar)*x0+sqrt(1-alpha_bar)*epsilon. It is not a trained reverse diffusion sampler.

Before running: predict the output or the invariant being checked.


In [4]:
x0=video[7]; noise=rng.normal(size=x0.shape)
for alpha_bar in [.99,.5,.01]:
    xt=np.sqrt(alpha_bar)*x0+np.sqrt(1-alpha_bar)*noise
    print("alpha_bar",alpha_bar,"MSE from clean",np.mean((xt-x0)**2))
for fps in [1,2,4]: print("30-second clip tokens at",fps,"fps:",30*fps*196)
assert 30*2*196==11760


alpha_bar 0.99 MSE from clean 0.010096291966042689
alpha_bar 0.5 MSE from clean 0.5122983970807312
alpha_bar 0.01 MSE from clean 1.051587278227348
30-second clip tokens at 1 fps: 5880
30-second clip tokens at 2 fps: 11760
30-second clip tokens at 4 fps: 23520


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why can good single-frame accuracy coexist with poor temporal reasoning?
2. How could transcript timestamps corrupt multimodal grounding?
3. Is forward noising alone a generative diffusion model?


## Extension and submission

Use the video-model pathway in the model-selection guide for a research extension only; freeze a short clip, frame timestamps, checkpoint revision, and generation budget before comparing systems.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
